# 🔍 Notebook: Retrieval-Augmented Generation (RAG) — *Part 1 of 3*

*This is the first of three notebooks in this chapter: 1) **2-Step RAG** → 2) Agentic RAG → 3) Hybrid RAG. They build on each other — start here.*

In this notebook we build the simplest and most common form of RAG: retrieval **always** happens before generation, in a fixed two-step pipeline. Along the way we introduce the fundamental building blocks (`Document`, loaders, text splitters, embeddings, vector stores, retrievers) that all three notebooks in this chapter rely on.

## 📚 Sources

- [LangChain Documentation: Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval)
- [LangChain Documentation: Build a Custom RAG Agent with LangGraph](https://docs.langchain.com/oss/python/langgraph/rag-agent)

## What is RAG, and why do we need it?

LLMs only know what was in their training data, up to some cutoff date. They can't answer questions about your company's internal documents, something that happened last week, or anything private. Worse, when they don't know something, they often **hallucinate** — confidently inventing a plausible-sounding but wrong answer instead of saying "I don't know".

**Retrieval-Augmented Generation (RAG)** fixes this by giving the model access to an external knowledge base at answer time: before generating a response, we *retrieve* the most relevant pieces of text from that knowledge base and insert them into the prompt as context. The model then answers based on that context instead of (only) its frozen training data.

There isn't just one way to build a RAG system, though. This chapter covers three architectures, one per notebook:

| # | Architecture | How retrieval is triggered | Notebook |
|---|---|---|---|
| 1 | **2-Step RAG** | Always: retrieve, then generate. Fixed pipeline, no decisions made at runtime. | *this notebook* |
| 2 | **Agentic RAG** | An LLM agent decides *if* and *when* to retrieve, by calling retrieval as a tool. | `07_2_agentic_rag.ipynb` |
| 3 | **Hybrid RAG** | Combines both: a structured pipeline with built-in validation steps (is the query good? are the retrieved documents actually relevant? is the answer any good?) that can rewrite the query or retry. | `07_3_hybrid_rag.ipynb` |

2-Step RAG is the simplest: fast, predictable, and easy to debug, since the same thing happens on every call. Its weakness is exactly that rigidity — it retrieves even when retrieval isn't needed, and it has no way to notice or recover from a bad retrieval. That's what the next two notebooks address.

## The building blocks of a retrieval pipeline

Every RAG system, regardless of architecture, is built from the same pieces:

```
raw data → Loader → Documents → Text Splitter → chunks (Documents) → Embedding Model → vectors → Vector Store → Retriever
```

1. **Loader** — reads your raw data (a PDF, a website, a database, a JSON file, ...) and turns it into `Document` objects.
2. **Text Splitter** — breaks long documents into smaller chunks, so they fit into an embedding model's input limit and so retrieval can be more precise.
3. **Embedding Model** — converts each chunk of text into a vector (a list of numbers) that captures its meaning.
4. **Vector Store** — stores all those vectors and can quickly find the ones most similar to a query vector.
5. **Retriever** — the interface you actually call: give it a query, it returns the most relevant `Document`s.

We'll go through each of these one at a time.

### The `Document` object

Almost everything in LangChain's retrieval tooling — loaders, splitters, vector stores, retrievers — passes data around as `Document` objects. A `Document` is a very simple container with just two things:

- `page_content`: a string, the actual text.
- `metadata`: a dictionary of arbitrary extra information about that text (e.g. a source file name, a page number, an author, a label, ...). Metadata isn't used for retrieval matching itself, but it's incredibly useful for filtering, citing sources, or displaying results.

You'll almost never need to build `Document`s by hand when loading files — loaders do that for you (see below) — but it's worth seeing the bare object once, since everything else in this chapter is built on top of it.

In [1]:
from langchain_core.documents import Document

example_doc = Document(
    page_content="RAG combines retrieval of external documents with LLM generation.",
    metadata={"source": "lecture_notes.txt", "topic": "RAG"},
)

print(example_doc)
print("\npage_content:", example_doc.page_content)
print("metadata:", example_doc.metadata)

page_content='RAG combines retrieval of external documents with LLM generation.' metadata={'source': 'lecture_notes.txt', 'topic': 'RAG'}

page_content: RAG combines retrieval of external documents with LLM generation.
metadata: {'source': 'lecture_notes.txt', 'topic': 'RAG'}


## 1. Loading documents into a knowledge base

LangChain ships loaders for many formats — a few examples:

```python
from langchain_community.document_loaders import PyPDFLoader     # PDFs
from langchain_community.document_loaders import TextLoader      # plain text files
from langchain_community.document_loaders import WebBaseLoader   # web pages
from langchain_community.document_loaders import DirectoryLoader # a whole folder of files
```

Every loader has a `.load()` method that returns a `list[Document]`. More loaders are listed in the [LangChain documentation](https://docs.langchain.com/oss/python/integrations/document_loaders).

But not all data comes as a file you can point a loader at. Structured data (a JSON export, an API response, a database table) usually needs to be turned into `Document`s yourself — which is just as easy, since a `Document` is nothing more than text + metadata.

As a running example for this whole chapter, we'll build a RAG system over **real customer tweets about airlines** — the same kind of data you already worked with in `02_intro_nlp_in_python.ipynb`'s sentiment analysis exercises, here used as a searchable knowledge base instead. Think of it as a support-team tool: "what are customers actually saying about X?"

We already drew a fixed random sample of 500 tweets (with labeled sentiment) from the [`iecjsu/airlineSFT_All`](https://huggingface.co/datasets/iecjsu/airlineSFT_All) dataset on Hugging Face and saved them to `content/airline_tweets.json`, so this notebook — and the two after it — always work with the exact same 500 tweets, instead of re-sampling (and getting different results) on every run.

In [2]:
import json

with open("content/airline_tweets.json") as f:
    tweets = json.load(f)

print(f"Number of tweets: {len(tweets)}")
print(json.dumps(tweets[1], indent=2, ensure_ascii=False))

Number of tweets: 500
{
  "text": "@united Hello we are doing a world record attempt on the amount of ball point pens in a collection please could you help with a pen?",
  "sentiment": "neutral"
}


In [3]:
from langchain_core.documents import Document

# One Document per tweet. page_content is the tweet text (what gets matched against),
# metadata keeps the sentiment label around for later.
docs = [Document(page_content=t["text"], metadata={"sentiment": t["sentiment"]}) for t in tweets]

print(f"Number of documents: {len(docs)}")
print("\nExample document:")
print(docs[1])

Number of documents: 500

Example document:
page_content='@united Hello we are doing a world record attempt on the amount of ball point pens in a collection please could you help with a pen?' metadata={'sentiment': 'neutral'}


### Splitting text into chunks

Embedding models have an input length limit (often somewhere around 512–8192 tokens, depending on the model), and retrieval works better when chunks are focused on one topic rather than spanning many. `RecursiveCharacterTextSplitter` handles this: it tries a list of separators from most to least "natural" (e.g. paragraph breaks, then line breaks, then spaces), and always uses the first one that gets each chunk under `chunk_size` characters.

`chunk_overlap` makes consecutive chunks share their last/first few characters, so information straddling a chunk boundary isn't lost.

Our tweets are already short, single-topic units of text (that's just what a tweet is) — splitting them further would make no sense, and indeed the longest one in our dataset is nowhere near typical embedding limits. So for our corpus we deliberately keep **one `Document` per tweet, unsplit**. But it's worth seeing the splitter actually split something, so let's run it on a longer, synthetic example paragraph as an illustration:

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

example_text = """Customer support at airlines handles a huge volume of feedback every day, ranging from
simple questions about baggage policy to urgent complaints about missed connections and cancelled flights.

Understanding this feedback at scale is hard: a support team cannot read every single tweet manually, and
keyword search alone often misses complaints phrased in unexpected ways. This is exactly the kind of problem
retrieval-augmented systems are built to help with, by letting a support agent ask a natural-language question
and get back a handful of genuinely relevant, real examples instead of skimming thousands of raw messages."""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,       # small on purpose, just to force a visible split
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_text(example_text)

print(f"Split into {len(chunks)} chunks.")
for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i + 1} ({len(c)} chars) ---\n{c}")

Split into 3 chunks.

--- Chunk 1 (193 chars) ---
Customer support at airlines handles a huge volume of feedback every day, ranging from
simple questions about baggage policy to urgent complaints about missed connections and cancelled flights.

--- Chunk 2 (214 chars) ---
Understanding this feedback at scale is hard: a support team cannot read every single tweet manually, and
keyword search alone often misses complaints phrased in unexpected ways. This is exactly the kind of problem

--- Chunk 3 (217 chars) ---
retrieval-augmented systems are built to help with, by letting a support agent ask a natural-language question
and get back a handful of genuinely relevant, real examples instead of skimming thousands of raw messages.


## 2. Keyword-based retrieval (BM25)

**BM25** is a classic ranking algorithm based purely on **keyword matching** — no embeddings, no LLM involved. It scores each document by how often and how distinctively the query's words appear in it, and returns the highest-scoring documents. It's fast, needs no extra model, and is very good when queries use the same vocabulary as the documents (e.g. exact names, codes, technical terms).

In [5]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 3  # how many documents to return

query = "friendly and helpful flight attendants"
results = bm25_retriever.invoke(query)

print(f"Query: {query!r}\n")
for d in results:
    print(f"- ({d.metadata['sentiment']}) {d.page_content}")

Query: 'friendly and helpful flight attendants'

- (positive) @united our flight attendant @superben was super helpful in finding a bag we left on a flight today. Excellent customer service. Name fits.
- (positive) @USAirways @AmericanAir Thank you for a couple of easy, hassle free flights today, professional and friendly staff made everything easy!
- (negative) @United is officially the worst, most delayed, and least helpful airline I have ever had the misfortune of flying on


/var/folders/qy/5gtwsk6s7jgbknbqgb533x9w0000gn/T/ipykernel_95711/453555446.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Notice the top BM25 result actually has **negative** sentiment — it matched on the words "flight attendants" but is a complaint, not praise. BM25 doesn't know what "friendly" and "helpful" *mean*, only that those exact words aren't in that tweet. Let's see if semantic search does better.

## 3. Semantic retrieval (embeddings)

**Embedding models** convert text into a vector that captures its *meaning* rather than its exact wording. You already saw this idea in notebook `02_intro_nlp_in_python.ipynb`, where spaCy's word vectors and `cosine_similarity` let us measure how semantically similar two single words are. Here we do the same thing, just with a model trained specifically for retrieval, and applied to whole chunks of text instead of single words: texts with similar meaning end up as nearby vectors, so "friendly" and "helpful" pull a tweet closer to our query even if those exact words don't appear in it.

A **vector store** holds all these vectors and can quickly find the ones closest to a query's vector (nearest-neighbor search). Wrapping a vector store `.as_retriever()` gives you the same `Retriever` interface as BM25 above — the two are interchangeable in a RAG pipeline.

### Caching embeddings

Embedding 500 tweets means 500 requests to the embedding model. That's not instant, and if we recomputed it every single time we re-run this notebook, we'd waste a lot of time (and load on the shared university server) for no reason, since the text — and therefore its embedding — never changes.

So we cache: every time we embed a piece of text, we save the resulting vector to a JSON file keyed by the text itself. Before embedding, we check the cache first. We do this by subclassing `OllamaEmbeddings` and overriding `embed_documents` — a plain example of inheritance (see notebook `01_intro_python.ipynb`): we reuse everything from the parent class and only change the one method that needs the caching behavior, via `super()`.

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads LLM_HOST from a .env file in the project root (see notebook 03 / setup.md)

LLM_HOST = os.environ["LLM_HOST"]  # the IP address you got in the lecture
LLM_URL = f"http://{LLM_HOST}:11434"
LLM_REASONING = "gemma4:26b"       # the reasoning MoE model - also supports tool calling
EMBEDDING_MODEL = "embeddinggemma" # a dedicated embedding model available on the university's Ollama server

In [7]:
import json
import os
from langchain_ollama import OllamaEmbeddings

CACHE_PATH = "content/embedding_cache.json"


class CachedOllamaEmbeddings(OllamaEmbeddings):
    # OllamaEmbeddings that caches every embedding it computes in a JSON file on disk.
    cache_path: str = CACHE_PATH

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        cache = json.load(open(self.cache_path)) if os.path.exists(self.cache_path) else {}

        texts_to_embed = [t for t in texts if t not in cache]
        if texts_to_embed:
            print(f"Embedding {len(texts_to_embed)} new text(s) via the API...")
            new_vectors = super().embed_documents(texts_to_embed)  # the real API call, only for cache misses
            for text, vector in zip(texts_to_embed, new_vectors):
                cache[text] = vector
            json.dump(cache, open(self.cache_path, "w"))
        else:
            print("All texts already in cache, no API call needed.")

        return [cache[t] for t in texts]


embeddings = CachedOllamaEmbeddings(model=EMBEDDING_MODEL, base_url=LLM_URL)

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

# InMemoryVectorStore is the simplest possible vector store: it keeps every vector in a
# Python dict and computes similarity by brute force. That's plenty fast for 500 documents,
# and since it's rebuilt fresh every run, there's no leftover state to clean up between runs
# (unlike a persistent database, where you'd have to make sure not to insert duplicates).
vectorstore = InMemoryVectorStore(embeddings)
vectorstore.add_documents(docs)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "friendly and helpful flight attendants"
results = vector_retriever.invoke(query)

print(f"Query: {query!r}\n")
for d in results:
    print(f"- ({d.metadata['sentiment']}) {d.page_content}")

Embedding 415 new text(s) via the API...


Embedding 1 new text(s) via the API...


Query: 'friendly and helpful flight attendants'

- (positive) @USAirways thanks for getting us on ur plane. Awesome flight attendant who is making us smile after difficult travel. #customerservice
- (positive) @JetBlue 2324 from Orlando to Dca ! And my awesome flight attendant is Robert!
- (negative) @united she was at the service desk at gate 21, and helped us find a flight to get us to our dest. on time when our flight got Cancelled Flightled


Semantic search finds genuinely positive tweets about flight attendants, even without an exact word match — the improvement we were missing from BM25. In practice, production RAG systems often combine both (hybrid search) — but for this course, we'll stick to using one retriever at a time to keep things clear.

## 4. The 2-Step RAG chain: retrieve, then generate

Now we wire a retriever together with an LLM using LangChain Expression Language (LCEL, the `|` operator you've already seen in earlier notebooks): retrieve documents for the question, format them into the prompt's context, then generate an answer. This *always* happens, for every question — that's what makes it "2-Step" RAG.

In [9]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0)

template = """Answer the question based only on the following customer tweets. If the tweets don't contain the answer, say you don't know.

Tweets:
{context}

Question: {question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)


def format_docs(retrieved_docs):
    return "\n\n".join(d.page_content for d in retrieved_docs)


# The pipeline:
# 1. vector_retriever finds the most relevant tweets for the question
# 2. format_docs turns them into one string
# 3. RunnablePassthrough() forwards the question unchanged
# 4. prompt fills in {context} and {question}
# 5. llm generates the final answer
rag_chain = (
    {"context": vector_retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [10]:
for question in [
    "What do customers complain about regarding baggage?",
    "What do customers say about flight delays?",
]:
    answer = rag_chain.invoke(question)
    print(f"Q: {question}\nA: {answer.content}\n")

Embedding 1 new text(s) via the API...


Q: What do customers complain about regarding baggage?
A: Customers complain about losing free checked bags, long wait times for bags (specifically one hour at EWR), and the lack of baggage attendants at EWR.

Embedding 1 new text(s) via the API...


Q: What do customers say about flight delays?
A: Customers express frustration with delays, noting that American Airlines delays are "right on cue," a Southwest flight (1028) was delayed by 1.5 hours as part of a pattern of weekly delays, and one customer is asking about the cause of a delay for Southwest flight 1836.



## What 2-Step RAG can't do

Try asking the chain something that has nothing to do with airline tweets, like "What's 12 times 7?" — it will *still* retrieve three (irrelevant) tweets and stuff them into the prompt, because retrieval isn't a decision here, it's a fixed step. The model usually still answers correctly despite the irrelevant context, but it does unnecessary work, and with a less capable model, irrelevant context can actively hurt the answer.

This is exactly the gap **Agentic RAG** closes, in the next notebook: instead of a fixed step, retrieval becomes a *tool* that the LLM itself decides whether to use.

## Exercise: Filter results by sentiment

Extend `vector_retriever` (or write a small wrapper function around it) so that it only returns tweets with `negative` sentiment. Then re-run the RAG chain with a question like *"What are customers unhappy about?"* and compare the answer to the unfiltered version.

*Hint: `InMemoryVectorStore`'s filter is a Python function `Callable[[Document], bool]`, not a dict — check `vectorstore.as_retriever(search_kwargs={"filter": ...})`.*

In [11]:
# Insert code here...

<details>
<summary><b>Show solution</b></summary>

```python
negative_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "filter": lambda doc: doc.metadata["sentiment"] == "negative"}
)

negative_chain = (
    {"context": negative_retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
)

answer = negative_chain.invoke("What are customers unhappy about?")
print(answer.content)
```

</details>

---

**Next up:** `07_2_agentic_rag.ipynb` — same knowledge base, but retrieval becomes a tool the agent decides whether to use.